<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Finetuning/blob/main/88%20aphrodite_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

네, Google Colab 환경에서 `aphrodite-engine`을 실행하는 것은 충분히 가능합니다.

Aphrodite Engine은 vLLM을 포크하여 만들어진 고성능 인퍼런스 엔진으로, PagedAttention과 연속 배칭(Continuous Batching) 등 vLLM의 핵심 최적화 기술을 Colab 환경에서도 동일하게 활용할 수 있습니다.

Colab의 기본 T4 GPU(16GB VRAM) 환경에서는 메모리 초과(OOM)를 방지하기 위해 `gpu_memory_utilization` 파라미터를 조절하고, 7B 이하의 경량 모델이나 양자화된 모델을 타겟팅하는 것이 좋습니다. 아래에 노트북의 개별 셀(Cell)에 복사하여 바로 실행할 수 있는 두 가지 구동 방식(오프라인 배치 추론, API 서버) 예제를 작성해 드립니다.

---

### 1. 환경 설정 및 패키지 설치

Colab의 첫 번째 셀에서 엔진을 설치합니다.

In [ ]:
# [Cell 1] Aphrodite Engine 설치
!pip install -U aphrodite-engine

---

### 2. 예제 A: Python API를 이용한 오프라인 배치 추론

데이터 파이프라인 구축이나 합성 데이터셋 생성 스크립트 작성 시 모델을 스크립트 내로 직접 로드하여 사용하는 방식입니다.

In [7]:
# Configure the system's dynamic linker to find CUDA libraries
!echo "/usr/local/cuda/lib64" | sudo tee /etc/ld.so.conf.d/cuda.conf > /dev/null
!sudo ldconfig

# --- Diagnostic Step: Check for libcudart.so.13 existence ---
# List libcudart.so* files in the CUDA library path to verify its presence and exact name
print("Checking for libcudart.so.13 in /usr/local/cuda/lib64:")
!ls -l /usr/local/cuda/lib64/libcudart.so*
# --- End Diagnostic Step ---

import os

# Although ldconfig is used, keep os.environ for other potential Python-level uses
# Ensure that the CUDA library path is in LD_LIBRARY_PATH.
# This is often needed when specific libraries like aphrodite-engine
# are compiled against a specific CUDA version and need the path
# explicitly set for dynamic loading.
cuda_lib_path = "/usr/local/cuda/lib64"
if cuda_lib_path not in os.environ.get('LD_LIBRARY_PATH', ''):
    os.environ['LD_LIBRARY_PATH'] = f"{cuda_lib_path}:{os.environ.get('LD_LIBRARY_PATH', '')}"
    print(f"Added {cuda_lib_path} to LD_LIBRARY_PATH (via os.environ)")

# [Cell 2] 오프라인 추론 스크립트
from aphrodite import LLM, SamplingParams

# Colab T4 GPU의 VRAM 여유를 두기 위해 85%만 할당합니다.
# 테스트를 위해 비교적 가벼운 Qwen 1.5B Instruct 모델을 사용합니다.
llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    dtype="float16"
)

# 샘플링 파라미터 설정
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=512
)

# 합성 데이터셋 생성 프롬프트 시나리오 예시
prompts = [
    "Generate a synthetic dataset scenario focusing on system 'load' and automated operational latency metrics.",
    "설비 자동화 제어 시스템의 응답 지연 시간을 평가하기 위한 엣지 디바이스 부하(load) 테스트 시나리오를 작성해 줘."
]

# 병렬 배치 추론 실행
outputs = llm.generate(prompts, sampling_params)

# 결과 출력
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"==========\n[Prompt]\n{prompt}\n\n[Generated]\n{generated_text}\n")

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_

ImportError: libcudart.so.13: cannot open shared object file: No such file or directory

---

### 3. 예제 B: OpenAI 호환 API 서버 기반 실행

외부 Multi-Agent 오케스트레이션 프레임워크나 로컬 Microservice 테스트 루프와 연동할 때 사용하는 방식입니다. 엔진을 백그라운드로 띄우고 REST API를 통해 통신합니다.

```bash
# [Cell 3] API 서버 백그라운드 실행
# (실행 후 모델 가중치가 VRAM에 적재될 때까지 약 1~2분 정도 대기합니다.)
!nohup aphrodite run Qwen/Qwen2.5-1.5B-Instruct \
  --host 127.0.0.1 \
  --port 2242 \
  --gpu-memory-utilization 0.85 \
  --max-model-len 4096 \
  --dtype float16 > aphrodite_server.log 2>&1 &

```

서버가 정상적으로 구동되었는지 확인하고, `requests` 라이브러리를 통해 API 엔드포인트를 호출하는 코드입니다.

In [ ]:
# [Cell 4] API 서버 호출 테스트
import requests
import time

# 서버 초기화 대기
time.sleep(15)

url = "http://127.0.0.1:2242/v1/chat/completions"
headers = {"Content-Type": "application/json"}
data = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "messages": [
        {"role": "system", "content": "You are a senior system architect."},
        {"role": "user", "content": "Explain the importance of measuring system load in distributed multi-agent workflows."}
    ],
    "max_tokens": 256
}

try:
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()
    print("[API Response]")
    print(result['choices'][0]['message']['content'])
except Exception as e:
    print(f"API Request Failed: {e}\n(서버가 아직 로드되지 않았을 수 있습니다. 로그를 확인하거나 잠시 후 다시 시도해 주세요.)")

> **Colab 구동 시 주요 팁**
> * 양자화된 모델(AWQ, GPTQ 등)을 로드하여 사용하실 경우, 파이썬 스크립트의 `LLM` 초기화 파라미터나 CLI 명령어에 `quantization="awq"`(또는 해당 포맷) 옵션을 추가하여 T4 환경 내 리소스 활용을 극대화할 수 있습니다.
>
>

In [8]:
print('Checking aphrodite-engine version:')
!pip show aphrodite-engine

Checking aphrodite-engine version:
Name: aphrodite-engine
Version: 0.21.0
Summary: Serving LLMs at Scale
Home-page: https://github.com/aphrodite-engine/aphrodite-engine
Author: PygmalionAI
Author-email: 
License: GNU AFFERO GENERAL PUBLIC LICENSE
                       Version 3, 19 November 2007

 Copyright (C) 2007 Free Software Foundation, Inc. <http://fsf.org/>
 Everyone is permitted to copy and distribute verbatim copies
 of this license document, but changing it is not allowed.

                            Preamble

  The GNU Affero General Public License is a free, copyleft license for
software and other kinds of works, specifically designed to ensure
cooperation with the community in the case of network server software.

  The licenses for most software and other practical works are designed
to take away your freedom to share and change the works.  By contrast,
our General Public Licenses are intended to guarantee your freedom to
share and change all versions of a program--to m

Once you have identified a version of `aphrodite-engine` that is compatible with CUDA 12 (e.g., `aphrodite-engine==X.Y.Z`), you can use the following commands to first uninstall the current version and then install the compatible one.

In [9]:
# Uninstall the current aphrodite-engine version
!pip uninstall -y aphrodite-engine

# Install the CUDA 12 compatible version of aphrodite-engine
# IMPORTANT: Replace 'X.Y.Z' with the actual version number you found to be compatible with CUDA 12.
# Example: !pip install aphrodite-engine==0.2.0 (This is a placeholder example, verify the actual version.)
!pip install aphrodite-engine==X.Y.Z

print("Please replace 'X.Y.Z' in the cell above with the actual version of aphrodite-engine compatible with CUDA 12.")
print("You may need to restart the runtime after installation if issues persist.")

Found existing installation: aphrodite-engine 0.21.0
Uninstalling aphrodite-engine-0.21.0:
  Successfully uninstalled aphrodite-engine-0.21.0
ERROR: Invalid requirement: 'aphrodite-engine==X.Y.Z': Expected end or semicolon (after name and no valid version specifier)
    aphrodite-engine==X.Y.Z
                    ^
Please replace 'X.Y.Z' in the cell above with the actual version of aphrodite-engine compatible with CUDA 12.
You may need to restart the runtime after installation if issues persist.


In [2]:
print('Checking CUDA version:')
!nvidia-smi

Checking CUDA version:
Mon Jul 13 23:46:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------